# Raccolta Dati da Reddit — r/Italia, keyword: *notizie, film, sport*
Raccolta tramite **Arctic Shift** (archivio pubblico Reddit per ricerca accademica) di:
- **100 post per keyword** (300 totali) di r/Italia
- e i **top 50 commenti** per ciascun post.

Le parole cercate in ogni post sono: **notizie, film, sport**.

## Installazione dipendenze

In [1]:
# !pip install requests spacy tqdm
# !python -m spacy download it_core_news_sm     # versione small
# !python -m spacy download it_core_news_md     # versione media (bilancio qualità/velocità)
# !python -m spacy download it_core_news_lg     # versione large (migliore, + lenta)

/Users/veronicabosso/PycharmProjects/ELIta_tesi/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.9/567.9 MB 5.3 MB/s  0:01:410:00:0100:03
✔ Download and installation successful
You can now load the package via spacy.load('it_core_news_lg')


## Configurazione

In [2]:
import requests
import time
import calendar
import pandas as pd
from datetime import datetime

SUBREDDIT         = "Italia"
KEYWORDS          = ["notizie", "film", "sport"]
N_POSTS_PER_KW    = 100
COMMENTS_PER_POST = 50    # quanti commenti tenere (top N per score)
FETCH_LIMIT       = 250   # quanti commenti recuperare dall'API prima di filtrare

BASE_URL = "https://arctic-shift.photon-reddit.com/api"
HEADERS  = {"User-Agent": "python:elita.tesi.multi:v1.0 (academic NLP)"}

print("Configurazione:")
print(f"  Subreddit             : r/{SUBREDDIT}")
print(f"  Keywords              : {KEYWORDS}")
print(f"  Post per keyword      : {N_POSTS_PER_KW}  (totale: {N_POSTS_PER_KW * len(KEYWORDS)})")
print(f"  Commenti recuperati   : {FETCH_LIMIT} (top {COMMENTS_PER_POST} per score)")

/Users/veronicabosso/PycharmProjects/ELIta_tesi/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Configurazione:
  Subreddit             : r/Italia
  Keywords              : ['notizie', 'film', 'sport']
  Post per keyword      : 100  (totale: 300)
  Commenti recuperati   : 250 (top 50 per score)


## Test connessione API
Attenzione, con servizi intermedi come **Arctic Shift** le risposte possono variare. Se lo status è 200 OK, altrimenti se è 422 riprova dopo qualche secondo.

In [3]:
# Test 1: endpoint posts/search (prima keyword)
resp = requests.get(f"{BASE_URL}/posts/search", headers=HEADERS, timeout=20, params={
    "subreddit": SUBREDDIT,
    "title": KEYWORDS[0],
    "limit": 2,             # risultati restituiti per chiamata
    "after":  "2025-01-01",
    "before": "2025-01-31"
})
print(f"[posts/search]    Status: {resp.status_code}")
if resp.status_code == 200 and resp.json().get("data"):
    p = resp.json()["data"][0]
    sample_post_id = p.get("id", "")
    print(f"  Titolo   : {p.get('title','')[:80]}")
    print(f"  Score    : {p.get('score',0)}  |  Commenti: {p.get('num_comments',0)}")
    print(f"  Post ID  : {sample_post_id}")
else:
    sample_post_id = ""
    print(f"  Errore: {resp.text[:200]}")

# Test 2: endpoint comments/search per un post specifico
if sample_post_id:
    resp2 = requests.get(f"{BASE_URL}/comments/search", headers=HEADERS, timeout=20, params={
        "link_id": sample_post_id,
        "limit": 3
    })
    print(f"\n[comments/search] Status: {resp2.status_code}")
    if resp2.status_code == 200 and resp2.json().get("data"):
        print(f"  Primo commento: {resp2.json()['data'][0].get('body','')[:150]}")
    else:
        print(f"  Errore: {resp2.text[:200]}")

[posts/search]    Status: 200
  Titolo   : Il Pil dell'Italia è fermo, la disoccupazione sale al 6,2% - Notizie
  Score    : 241  |  Commenti: 120
  Post ID  : 1idmjsb

[comments/search] Status: 200
  Primo commento: [removed]


## Raccolta Post

In r/Italia si vogliono ottenere i post con keyword = [notizie, film, sport] nel **titolo** tramite `/api/posts/search`.

Finestre mensili per evitare il rate limit (stessa tecnica dei commenti).

In [4]:
def collect_posts(subreddit, keyword, target=100):
    all_posts = []

    time_windows = []
    for year in [2025, 2024, 2023]:
        max_month = 12
        for month in range(max_month, 0, -1):
            last_day = calendar.monthrange(year, month)[1]
            time_windows.append((f"{year}-{month:02d}-01", f"{year}-{month:02d}-{last_day}", f"{year}-{month:02d}"))

    for after, before, label in time_windows:
        if len(all_posts) >= target:
            break

        params = {"subreddit": subreddit, "title": keyword, "limit": 25, "after": after, "before": before}
        last_utc = None

        while len(all_posts) < target:
            if last_utc:
                params["before"] = last_utc
            try:
                resp = requests.get(f"{BASE_URL}/posts/search", headers=HEADERS, params=params, timeout=30)
                if resp.status_code == 422:
                    break
                resp.raise_for_status()
                items = resp.json().get("data", [])
            except Exception as e:
                print(f"  Errore {label}: {e}")
                break

            if not items:
                break

            for p in items:
                if len(all_posts) >= target:
                    break
                title = p.get("title", "").strip()
                if not title:
                    continue
                selftext = p.get("selftext", "").strip()
                if selftext in ("[removed]", "[deleted]"):
                    selftext = ""
                ts = float(p.get("created_utc", 0))
                all_posts.append({
                    "post_id"     : p.get("id", ""),
                    "title"       : title,
                    "selftext"    : selftext,
                    "author"      : p.get("author", ""),
                    "timestamp"   : datetime.utcfromtimestamp(ts).isoformat(),
                    "score"       : p.get("score", 0),
                    "num_comments": p.get("num_comments", 0),
                    "url"         : "https://reddit.com" + p.get("permalink", "") if p.get("permalink") else "",
                })

            oldest_utc = min(float(p.get("created_utc", 0)) for p in items)
            last_utc   = datetime.utcfromtimestamp(oldest_utc).strftime("%Y-%m-%dT%H:%M:%S")
            if len(items) < 50:
                break
            time.sleep(1)

        print(f"  [{keyword}] {label}: {len(all_posts)} post totali", end="\r")
        time.sleep(1.5)

    print(f"\n  [{keyword}] Post raccolti: {len(all_posts)}")
    return all_posts

# Raccolta per tutte le keyword
all_posts = []
for kw in KEYWORDS:
    print(f"\n--- Raccolta keyword: '{kw}' ---")
    posts_kw = collect_posts(SUBREDDIT, kw, target=N_POSTS_PER_KW)
    for p in posts_kw:
        p["keyword"] = kw
    all_posts.extend(posts_kw)


--- Raccolta keyword: 'notizie' ---
  [notizie] 2024-11: 100 post totali
  [notizie] Post raccolti: 100

--- Raccolta keyword: 'film' ---
  [film] 2025-06: 100 post totali
  [film] Post raccolti: 100

--- Raccolta keyword: 'sport' ---
  [sport] 2023-10: 100 post totali
  [sport] Post raccolti: 100


In [5]:
df_posts = pd.DataFrame(all_posts)
print(f"\nPost totali raccolti: {len(df_posts)}")
print(df_posts.groupby("keyword").size().rename("n_post"))


Post totali raccolti: 300
keyword
film       100
notizie    100
sport      100
Name: n_post, dtype: int64


## Raccolta Commenti per Post
Per ogni post, si vogliono raccoglire fino a **100 commenti** dall'API (ordinati per `created_utc` discendente), poi li si ordinerà per score decrescente tenedo i **top 50**.

Questo garantisce che i commenti selezionati siano quelli più apprezzati dalla community e quindi che rappresentano meglio la reazione collettiva al post.

In [6]:
### ATTENZIONE non eseguire se non vuoi perdere 10 min di vita !!!!!!!!!!!!!!
def collect_comments_for_posts(df_posts, n_comments=50, fetch_limit=250):
    """
    Per ogni post:
      - recupera fino a `fetch_limit` commenti dall'API paginando correttamente
      - ordina per score decrescente
      - tiene i top `n_comments`
    """
    all_comments = []

    for i, row in df_posts.iterrows():
        post_id  = row["post_id"]
        raw_items = []
        last_utc  = None

        # Paginazione: continua finché non raggiungiamo fetch_limit o l'API non ha più risultati
        while len(raw_items) < fetch_limit:
            params = {"link_id": post_id, "limit": 100}
            if last_utc:
                params["before"] = last_utc

            try:
                resp = requests.get(f"{BASE_URL}/comments/search", headers=HEADERS,
                                    params=params, timeout=30)
                if resp.status_code != 200:
                    break
                items = resp.json().get("data", [])
            except Exception as e:
                print(f"  Errore post {post_id}: {e}")
                break

            if not items:
                break

            raw_items.extend(items)

            # Aggiorna il cursore PRIMA di decidere se continuare
            oldest_utc = min(float(c.get("created_utc", 0)) for c in items)
            last_utc   = datetime.utcfromtimestamp(oldest_utc).strftime("%Y-%m-%dT%H:%M:%S")

            if len(items) < 100:    # Se l'API ha restituito meno di 100, non ci sono altre pagine
                break

            time.sleep(0.3)

        # Filtra rimossi/troppo corti, ordina per score, tieni i top n
        valid = [
            c for c in raw_items
            if c.get("body", "").strip() not in ("", "[deleted]", "[removed]")
            and len(c.get("body", "").strip()) >= 10
        ]
        top = sorted(valid, key=lambda c: c.get("score", 0), reverse=True)[:n_comments]

        for rank, c in enumerate(top, start=1):
            ts = float(c.get("created_utc", 0))
            all_comments.append({
                "comment_id"   : c.get("id", ""),
                "post_id"      : post_id,
                "rank_by_score": rank,
                "comment_text" : c.get("body", "").strip(),
                "author"       : c.get("author", ""),
                "timestamp"    : datetime.utcfromtimestamp(ts).isoformat(),
                "score"        : c.get("score", 0),
                "permalink"    : "https://reddit.com" + c.get("permalink", "")
                                  if c.get("permalink") else "",
            })

        print(f"  [{i+1:3d}/{len(df_posts)}] {post_id}: "
              f"{len(raw_items)} recuperati → {len(valid)} validi → top {len(top)}")
        time.sleep(0.8)

    print(f"\nCommenti totali: {len(all_comments)}")
    return all_comments

comments = collect_comments_for_posts(df_posts, n_comments=COMMENTS_PER_POST, fetch_limit=FETCH_LIMIT)
df_comments = pd.DataFrame(comments)
print(df_comments[["comment_id", "post_id", "rank_by_score", "score", "comment_text"]].head(8))

  [  1/300] 1pxwodz: 12 recuperati → 12 validi → top 12
  [  2/300] 1pxsmol: 92 recuperati → 89 validi → top 50
  [  3/300] 1pwtbgw: 1 recuperati → 1 validi → top 1
  [  4/300] 1pto2h7: 4 recuperati → 4 validi → top 4
  [  5/300] 1pqrs65: 1 recuperati → 1 validi → top 1
  [  6/300] 1plic73: 1 recuperati → 1 validi → top 1
  [  7/300] 1pjjn8z: 4 recuperati → 4 validi → top 4
  [  8/300] 1piwsu5: 1 recuperati → 1 validi → top 1
  [  9/300] 1p8p9mf: 196 recuperati → 191 validi → top 50
  [ 10/300] 1p417zm: 6 recuperati → 6 validi → top 6
  [ 11/300] 1oxlmvh: 1 recuperati → 1 validi → top 1
  [ 12/300] 1osphwc: 105 recuperati → 101 validi → top 50
  [ 13/300] 1oqe4py: 3 recuperati → 3 validi → top 3
  [ 14/300] 1oj7eae: 2 recuperati → 2 validi → top 2
  [ 15/300] 1oaokd1: 187 recuperati → 184 validi → top 50
  [ 16/300] 1o9znmz: 143 recuperati → 141 validi → top 50
  [ 17/300] 1o8d128: 10 recuperati → 10 validi → top 10
  [ 18/300] 1o681ix: 22 recuperati → 20 validi → top 20
  [ 19/300] 1o

## Salvataggio CSV
- `posts_Italia_multi.csv` — un post per riga, con colonna `keyword`
- `comments_Italia_multi.csv` — un commento per riga, con `post_id` come chiave di join
- `corpus_Italia_multi.csv` — long format: una riga per documento (post o commento), con `keyword` per filtrare per topic

In [21]:
df_posts.to_csv("posts_Italia_multi.csv", index=False, encoding="utf-8-sig")
print(f"Post salvati    : {len(df_posts)} righe → 'posts_Italia_multi.csv'")

df_comments.to_csv("comments_Italia_multi.csv", index=False, encoding="utf-8-sig")
print(f"Commenti salvati: {len(df_comments)} righe → 'comments_Italia_multi.csv'")

Post salvati    : 300 righe → 'posts_Italia_multi.csv'
Commenti salvati: 4299 righe → 'comments_Italia_multi.csv'


In [22]:
# Corpus long-format con colonna keyword
posts_long = df_posts.assign(
    type="post",
    doc_id=df_posts["post_id"],
    text=df_posts.apply(
        lambda r: (r["title"] + "\n" + r["selftext"]).strip() if r["selftext"] else r["title"],
        axis=1
    ),
    rank_by_score=None,
)[["doc_id", "post_id", "keyword", "type", "text", "author", "timestamp", "score", "rank_by_score"]]

comments_long = df_comments.merge(df_posts[["post_id", "keyword"]], on="post_id", how="left") \
    .rename(columns={"comment_id": "doc_id", "comment_text": "text"}) \
    .assign(type="comment")[["doc_id", "post_id", "keyword", "type", "text", "author", "timestamp", "score", "rank_by_score"]]

df_corpus = pd.concat([posts_long, comments_long], ignore_index=True)
df_corpus.to_csv("corpus_Italia_multi.csv", index=False, encoding="utf-8-sig")
print(f"Corpus salvato  : {len(df_corpus)} righe → 'corpus_Italia_multi.csv'")

Corpus salvato  : 4599 righe → 'corpus_Italia_multi.csv'


In [8]:
print(df_corpus.groupby(["keyword", "type"]).size().unstack(fill_value=0))

type     comment  post
keyword               
film         930   100
notizie     1842   100
sport       1527   100


## Statistiche del corpus

In [9]:
import plotly.express as px

print("=" * 50)
print("STATISTICHE CORPUS")
print("=" * 50)
print(f"Post totali              : {len(df_posts)}")
print(f"  - con testo (selftext) : {(df_posts['selftext'] != '').sum()}")
print(f"  - solo link            : {(df_posts['selftext'] == '').sum()}")
print(f"Commenti totali          : {len(df_comments)}")
print(f"Media commenti per post  : {len(df_comments)/len(df_posts):.1f}")

STATISTICHE CORPUS
Post totali              : 300
  - con testo (selftext) : 156
  - solo link            : 144
Commenti totali          : 4299
Media commenti per post  : 14.3


In [10]:
# Post per mese
df_posts["mese"] = df_posts["timestamp"].str[:7]
fig1 = px.bar(df_posts["mese"].value_counts().sort_index().reset_index(),
              x="mese", y="count", title="Post per mese",
              labels={"mese": "Mese", "count": "Post"})
fig1.show()

In [11]:
# Distribuzione commenti per post
commenti_per_post = df_comments.groupby("post_id").size().reset_index(name="n_commenti")
fig2 = px.histogram(commenti_per_post, x="n_commenti", nbins=26,
                    title="Distribuzione commenti raccolti per post",
                    labels={"n_commenti": "Commenti raccolti", "count": "Post"})
fig2.show()

In [12]:
print("\nTop 10 post per score:")
display(df_posts.nlargest(10, "score")[["title", "score", "num_comments", "timestamp"]])


Top 10 post per score:


,title,score,num_comments,timestamp
37,Le notizie belle,787,99,2025-09-04T19:07:22
131,"Ho appena rivisto il grandissimo film ""Mrs. Do...",504,41,2025-11-09T22:35:36
201,In Italia esiste solo il calcio? La fatica di ...,489,437,2025-12-07T11:50:29
221,Il tennis potrebbe diventare lo sport più prat...,343,177,2025-07-18T04:45:23
169,Ci avete mai fatto caso che il Salento visto d...,311,19,2025-08-20T14:30:37
14,Il governo Meloni è da oggi il terzo più longe...,269,182,2025-10-19T12:37:33
102,"Buen camino, merita? È un bel film?",268,139,2025-12-28T14:39:25
71,Notizie top tier,262,54,2025-05-02T12:03:46
50,La Ferrero vicina ad acquistare il colosso Kel...,229,46,2025-07-09T21:34:38
80,Le grandi notizie di Repubblica,211,25,2025-03-16T23:13:06


## Tokenizzazione, Lemmatizzazione e POS-tagging

Verrà utilizzato il modello italiano `it_core_news_lg` di spaCy per analizzare tutti i documenti del corpus (post e commenti). Per ogni token estraiamo:
- `token`: testo originale
- `lemma`: forma base in minuscolo
- `pos`: parte del discorso (NOUN, VERB, ADJ, EMOJI, ecc.)

Filtriamo spazi e punteggiatura. L'identificatore è `doc_id` (vale sia per post che commenti).

Salviamo il risultato in `tokens_Italia_notizie.csv`.

In [17]:
import spacy
import emoji as emoji_lib
from tqdm.auto import tqdm

nlp = spacy.load("it_core_news_lg")

def process_text(text):
    doc = nlp(str(text))
    tokens = []
    for token in doc:
        if token.is_space or token.is_punct:
            continue
        pos = "EMOJI" if emoji_lib.emoji_count(token.text) > 0 else token.pos_
        tokens.append({
            "token" : token.text,
            "lemma" : token.lemma_.lower(),
            "pos"   : pos,
        })
    return tokens

In [32]:
print(f"Processamento di {len(df_corpus)} documenti con spaCy...")

all_tokens = []
for _, row in tqdm(df_corpus.iterrows(), total=len(df_corpus)):
    for t in process_text(row["text"]):
        all_tokens.append({"doc_id": row["doc_id"], "keyword": row["keyword"], **t})

df_tokens = pd.DataFrame(all_tokens)

print(df_tokens.groupby("keyword").size().rename("n_token"))

Processamento di 4599 documenti con spaCy...


100%|██████████| 4599/4599 [00:41<00:00, 111.28it/s]


keyword
film       40188
notizie    68354
sport      66829
Name: n_token, dtype: int64


In [34]:
df_tokens.to_csv("tokens_Italia_multi.csv", index=False, encoding="utf-8-sig")
print(f"\nToken salvati in 'tokens_Italia_multi.csv'")


Token salvati in 'tokens_Italia_multi.csv'


In [33]:
print(df_tokens['pos'].value_counts().to_string())

pos
NOUN     35295
ADP      23932
VERB     23474
DET      18074
ADV      16180
PRON     12518
ADJ      11612
AUX      10060
PROPN     7483
CCONJ     7008
SCONJ     4619
NUM       2329
X         1656
PUNCT      413
INTJ       403
EMOJI      166
SYM        120
PART        29


## spaCy su ELIta

Si applica spaCy su tutte le parole di ELIta per vedere come vengono POS-taggati i lemmi di ELIta.

In [26]:
df_elita_raw = pd.read_csv("../Fase1/ELIta_INTENSITY_Matrix.csv", index_col=0)
elita_words  = df_elita_raw.index.tolist()

print(f"Parole in ELIta: {len(elita_words)}")

Parole in ELIta: 6905


In [28]:
elita_pos_rows = []
for word in tqdm(elita_words):
    tokens = process_text(word)
    if not tokens:
        continue
    main_tok = tokens[0]    # Per parole multi-token (es. "a_malincuore") prendiamo il POS del primo token
    elita_pos_rows.append({
        "elita_word"  : word,
        "spacy_token" : main_tok["token"],
        "spacy_lemma" : main_tok["lemma"],
        "pos"         : main_tok["pos"],
        "n_tokens"    : len(tokens),   # quanti token spaCy produce (1 = parola singola)
    })

df_elita_pos = pd.DataFrame(elita_pos_rows)

100%|██████████| 6905/6905 [00:16<00:00, 412.54it/s]


In [35]:
df_elita_pos.to_csv("tokens_ELIta.csv", index=False, encoding="utf-8-sig")
print(f"\nToken salvati in 'tokens_ELIta.csv'")


Token salvati in 'tokens_ELIta.csv'


In [36]:
# --- distribuzione POS ---
print("=== Distribuzione POS nelle parole ELIta ===")
pos_counts = df_elita_pos["pos"].value_counts()
total = len(df_elita_pos)
for pos, n in pos_counts.items():
    print(f"  {pos:<10} {n:>5}  ({n/total*100:.1f}%)")

=== Distribuzione POS nelle parole ELIta ===
  NOUN        3297  (47.8%)
  VERB        1793  (26.0%)
  ADJ         1336  (19.4%)
  ADV          221  (3.2%)
  EMOJI        184  (2.7%)
  PROPN         36  (0.5%)
  PRON          11  (0.2%)
  ADP            6  (0.1%)
  X              5  (0.1%)
  AUX            5  (0.1%)
  DET            3  (0.0%)
  SCONJ          2  (0.0%)
  NUM            2  (0.0%)
  PUNCT          1  (0.0%)


In [37]:
print("\n=== Esempi per categoria POS ===")
for pos in pos_counts.index:
    sample = df_elita_pos[df_elita_pos["pos"] == pos]["elita_word"].head(8).tolist()
    print(f"  {pos:<10}: {', '.join(sample)}")


=== Esempi per categoria POS ===
  NOUN      : a_caso, a_malincuore, abbandono, abbellimento, abbigliamento, abbondanza, abietto, abiezione
  VERB      : a_scanso_di, abbandonare, abbandonato, abbassare, abbattere, abbattersi, abbattuto, abbinare
  ADJ       : abbagliante, abbonamento, abbondante, abbordabile, aberrante, abile, abituale, abominevole
  ADV       : abbastanza, accanto, addirittura, adesso, affatto, allora, alquanto, altrettanto
  EMOJI     : ☠, ☹, ☺, ♥️, ⚰️, ✈, ✊, ✌
  PROPN     : anfetamine, angelica, babele, bosco, buonasera, cagnesco, colombo, conterraneo
  PRON      : altro, cosa, divino, giornaliero, ingiusto, nessuno, niente, non_valido
  ADP       : call, hic_et_nunc, malgrado, remoto, senz'anima, serpente_a_sonagli
  X         : ago, down, psyco, team, twitter
  AUX       : astenersi_da, dovuto, leone_da_tastiera, san_valentino, voluto
  DET       : mio, opportuna, poco
  SCONJ     : quando, ventre
  NUM       : quattordici, ventiquattro
  PUNCT     : alter_ego


NOUN = nomi, ADJ = aggettivi, VERB = verbi, AUX = ausiliari, EMOJI = emoji.

ADP = preposizioni, CCONJ = congiunzioni coordinanti, SCONJ = congiunzioni subordinanti, DET = determinatori, PRON = pronomi, ADV = avverbi, PROPN = nomi propri, NUM = numeri, PART = particelle, INTJ = interiezioni, PUNCT = punteggiatura.

---